**Preparação do Ambiente**

In [ ]:
!pip install -q roboflow segmentation-models
# Define que usaremos o Keras do TensorFlow
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 106.2 MB/s eta 0:00:00
Segmentation Models: using `tf.keras` framework.


**Conexão do Dataset do Roboflow**

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(2) # Usando a v2 que você setou
dataset_roboflow = version.download("png-mask-semantic")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tcc---modelo-segformer-2 in png-mask-semantic:: 100%|██████████| 637/637 [00:00<00:00, 3358.97it/s]


**Criando o Carregador de Dados (Data Generator)**

In [ ]:
import tensorflow as tf
import cv2
import numpy as np

def load_data(img_path, mask_path, size=(256, 256)):
    # Carrega imagem
    img = cv2.imread(img_path)
    img = cv2.resize(img, size)
    img = img / 255.0 # Normalização

    # Carrega máscara (em escala de cinza)
    mask = cv2.imread(mask_path, 0)
    mask = cv2.resize(mask, size)
    mask = np.expand_dims(mask, axis=-1)
    mask = (mask > 0).astype(np.float32) # Garante que seja 0 ou 1

    return img, mask

# Aqui você criaria um gerador (tf.data.Dataset) apontando para
# dataset.location + '/train/images' e '/train/masks'

**Definindo o Modelo (Baseado nos seus Slides)**

In [ ]:
BACKBONE = 'resnet34'  # ResNet é mais estável para download automático
preprocess_input = sm.get_preprocessing(BACKBONE)

model = sm.Unet(
    BACKBONE,
    encoder_weights='imagenet',
    classes=1,
    activation='sigmoid',
    input_shape=(256, 256, 3)
)

model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss, # Ótimo para detectar obstáculos
    metrics=[sm.metrics.iou_score]   # Métrica de precisão de sobreposição
)

85521592/85521592 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


**Treinamento**

In [ ]:
import glob
import os

def get_dataset_single_folder(base_dir, size=(256, 256), batch_size=8):
    # Pega todos os arquivos de imagem
    all_files = sorted(glob.glob(os.path.join(base_dir, "*.jpg")))

    # No Roboflow 'png-mask-semantic', as máscaras costumam ser .png
    # e as imagens originais .jpg. Vamos separar:
    img_list = [f for f in all_files if "_mask" not in f]
    mask_list = [f.replace(".jpg", "_mask.png") for f in img_list]

    # Verificação rápida: a máscara existe?
    final_imgs = []
    final_masks = []
    for i, m in zip(img_list, mask_list):
        if os.path.exists(m):
            final_imgs.append(i)
            final_masks.append(m)

    print(f"Encontrados {len(final_imgs)} pares de imagem/máscara em: {base_dir}")

    def generator():
        for img_path, mask_path in zip(final_imgs, final_masks):
            img, mask = load_data(img_path, mask_path, size)
            yield img, mask

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(size[0], size[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(size[0], size[1], 1), dtype=tf.float32)
        )
    )

    steps = max(1, len(final_imgs) // batch_size)
    return dataset.repeat().batch(batch_size).prefetch(tf.data.AUTOTUNE), steps

# Configurando os caminhos baseados no seu print
base_path = dataset_roboflow.location # pasta: tcc---modelo-segformer-2
train_dataset, train_steps = get_dataset_single_folder(os.path.join(base_path, "train"))
val_dataset, val_steps = get_dataset_single_folder(os.path.join(base_path, "valid"))

In [ ]:
# Cálculo correto
total_imagens = 316
batch_size = 8
passos_por_epoca = total_imagens // batch_size # Resultado: 39

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    steps_per_epoch=passos_por_epoca, # Isso limita a época a 39 passos
    validation_steps=10                # Limita a validação também
)

Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 67s 280ms/step - iou_score: 0.2267 - loss: 0.9629 - val_iou_score: 0.0475 - val_loss: 9.5783
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 189ms/step - iou_score: 0.4909 - loss: 0.5868 - val_iou_score: 0.0389 - val_loss: 2.2898
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 7s 192ms/step - iou_score: 0.6309 - loss: 0.4311 - val_iou_score: 0.0636 - val_loss: 1.6310
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 193ms/step - iou_score: 0.6757 - loss: 0.3860 - val_iou_score: 0.0620 - val_loss: 1.2444
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 198ms/step - iou_score: 0.7282 - loss: 0.3262 - val_iou_score: 0.0080 - val_loss: 1.2875
Epoch 6/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 204ms/step - iou_score: 0.7513 - loss: 0.2989 - val_iou_score: 6.1290e-04 - val_loss: 1.3468
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 207ms/step - iou_score: 0.7509 - loss: 0.2967 - val_iou_score: 6.6043e-04 - val_loss: 1.3455
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 212ms/step - iou_score: 0.7762 - loss: 0

In [ ]:
model.save("modelo_unet_tcc.h5")
print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!


Pegar as métricas que faltam (Latência e FPS)

In [ ]:
import time
import numpy as np

# 1. Garantir que o modelo está em modo de inferência
# O input_shape que você definiu foi (256, 256, 3)
dummy_input = np.random.rand(1, 256, 256, 3).astype(np.float32)

print("Iniciando medição de performance...")

# 2. Warm-up (Aquecimento)
# Importante para que o TensorFlow otimize o grafo e a GPU saia do estado de repouso
for _ in range(20):
    _ = model.predict(dummy_input, verbose=0)

# 3. Medição Real
repetitions = 100
start_time = time.time()

for _ in range(repetitions):
    _ = model.predict(dummy_input, verbose=0)

end_time = time.time()

# 4. Cálculos
total_time = (end_time - start_time) # em segundos
avg_latencia_ms = (total_time / repetitions) * 1000
fps = 1 / (total_time / repetitions)

print("-" * 30)
print(f"⏱️ Latência Média: {avg_latencia_ms:.2f} ms")
print(f"🚀 FPS (Frames por Segundo): {fps:.2f}")
print("-" * 30)

# **Código Completo**

In [ ]:
!pip install -q roboflow segmentation-models
# Define que usaremos o Keras do TensorFlow
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm






from roboflow import Roboflow

rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(2) # Usando a v2 que você setou
dataset_roboflow = version.download("png-mask-semantic")







import tensorflow as tf
import cv2
import numpy as np

def load_data(img_path, mask_path, size=(256, 256)):
    # Carrega imagem
    img = cv2.imread(img_path)
    img = cv2.resize(img, size)
    img = img / 255.0 # Normalização

    # Carrega máscara (em escala de cinza)
    mask = cv2.imread(mask_path, 0)
    mask = cv2.resize(mask, size)
    mask = np.expand_dims(mask, axis=-1)
    mask = (mask > 0).astype(np.float32) # Garante que seja 0 ou 1

    return img, mask

# Aqui você criaria um gerador (tf.data.Dataset) apontando para
# dataset.location + '/train/images' e '/train/masks'






BACKBONE = 'resnet34'  # ResNet é mais estável para download automático
preprocess_input = sm.get_preprocessing(BACKBONE)

model = sm.Unet(
    BACKBONE,
    encoder_weights='imagenet',
    classes=1,
    activation='sigmoid',
    input_shape=(256, 256, 3)
)

model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss, # Ótimo para detectar obstáculos
    metrics=[sm.metrics.iou_score]   # Métrica de precisão de sobreposição
)








import glob
import os

def get_dataset_single_folder(base_dir, size=(256, 256), batch_size=8):
    # Pega todos os arquivos de imagem
    all_files = sorted(glob.glob(os.path.join(base_dir, "*.jpg")))

    # No Roboflow 'png-mask-semantic', as máscaras costumam ser .png
    # e as imagens originais .jpg. Vamos separar:
    img_list = [f for f in all_files if "_mask" not in f]
    mask_list = [f.replace(".jpg", "_mask.png") for f in img_list]

    # Verificação rápida: a máscara existe?
    final_imgs = []
    final_masks = []
    for i, m in zip(img_list, mask_list):
        if os.path.exists(m):
            final_imgs.append(i)
            final_masks.append(m)

    print(f"Encontrados {len(final_imgs)} pares de imagem/máscara em: {base_dir}")

    def generator():
        for img_path, mask_path in zip(final_imgs, final_masks):
            img, mask = load_data(img_path, mask_path, size)
            yield img, mask

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(size[0], size[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(size[0], size[1], 1), dtype=tf.float32)
        )
    )

    steps = max(1, len(final_imgs) // batch_size)
    return dataset.repeat().batch(batch_size).prefetch(tf.data.AUTOTUNE), steps

# Configurando os caminhos baseados no seu print
base_path = dataset_roboflow.location # pasta: tcc---modelo-segformer-2
train_dataset, train_steps = get_dataset_single_folder(os.path.join(base_path, "train"))
val_dataset, val_steps = get_dataset_single_folder(os.path.join(base_path, "valid"))








# Cálculo correto
total_imagens = 316
batch_size = 8
passos_por_epoca = total_imagens // batch_size # Resultado: 39

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    steps_per_epoch=passos_por_epoca, # Isso limita a época a 39 passos
    validation_steps=10                # Limita a validação também
)






model.save("modelo_unet_tcc.h5")
print("Modelo salvo com sucesso!")